# Framework: Multiple Strands Per Port/Edge

This notebook explains a clean framework for moving from a **single global multiplicity** to **edge-specific multiplicities** in your topology generation pipeline.

Goal: keep your current architecture (path generation -> multiplicity search -> conjugacy -> symmetry dedup), but replace scalar `N` logic with a per-edge target map.

## 1) Core Idea in Words

### Old model
- One scalar `N` means every edge must be used exactly `N` times.
- Constraint per edge `e`: `sum_{p: e in p} m_p = N`.

### New model
- Use an edge-target dictionary `T[e]` (or list indexed by edge id).
- Constraint per edge `e`: `sum_{p: e in p} m_p = T[e]`.
- This supports heterogeneous requirements, e.g. `{0:2, 1:3, 2:2, 3:3, 4:0, 5:0, 6:0}`.

### Why this is the right abstraction
- `generate_valid_multiplicities` already enforces edge-balance constraints; it just assumes one shared target.
- Replacing scalar target with per-edge target gives you different strand counts naturally.
- Conjugacy and symmetry filtering can stay unchanged because they operate on resulting collections of paths, not on the type of target representation.

In [ ]:
from collections import Counter

def normalize_edge_targets(edge_ids, edge_targets, default_edge_target=0):
    """
    Convert edge_targets input to dict {edge_id: target}.

    Supported inputs:
    - int: same target for all edges
    - list/tuple: position i is target for edge i (must cover all edges)
    - dict: partial/full mapping; missing edges get default_edge_target
    """
    edge_ids = list(edge_ids)

    if isinstance(edge_targets, int):
        return {e: edge_targets for e in edge_ids}

    if isinstance(edge_targets, (list, tuple)):
        if len(edge_targets) != len(edge_ids):
            raise ValueError(
                f"Expected {len(edge_ids)} targets, got {len(edge_targets)}"
            )
        return {e: int(edge_targets[i]) for i, e in enumerate(edge_ids)}

    if isinstance(edge_targets, dict):
        return {e: int(edge_targets.get(e, default_edge_target)) for e in edge_ids}

    raise TypeError(
        "edge_targets must be int, list/tuple, or dict"
    )

def condition_fn_per_edge(collection, edge_ids, edge_targets):
    """Check exact edge balance for a candidate collection."""
    counts = Counter({e: 0 for e in edge_ids})
    for path, mult in collection.items():
        for e in path:
            counts[e] += mult
    return all(counts[e] == edge_targets[e] for e in edge_ids)

## 2) Main Rewrite: `generate_valid_multiplicities`

### What must change
- Replace scalar `max_multiplicity_edge` with dict `edge_targets`.
- At each recursion step, compute max allowed multiplicity from residual capacities of edges in current path.
- Final acceptance: all residuals exactly zero (equivalent to all counts meeting targets).

### Important bug to avoid
- Do not use fixed index access like `max_multiplicity_edge[2]`.
- Use `edge_targets[e]` for each edge `e`.

In [ ]:
def generate_valid_multiplicities_v2(sequences, edge_targets, edge_ids, max_multiplicity_path=None):
    """
    Yield tuples of path multiplicities that satisfy per-edge target usage exactly.

    sequences: list[tuple[int]]
    edge_targets: dict[int, int]
    edge_ids: iterable of all edge ids
    max_multiplicity_path: optional cap per path
    """
    if max_multiplicity_path is None:
        max_multiplicity_path = max(edge_targets.values()) if edge_targets else 0

    edge_ids = list(edge_ids)
    residual = {e: edge_targets[e] for e in edge_ids}
    n = len(sequences)

    def backtrack(i, current):
        if i == n:
            if all(residual[e] == 0 for e in edge_ids):
                yield tuple(current)
            return

        used_edges = set(sequences[i])
        max_allowed = min([max_multiplicity_path] + [residual[e] for e in used_edges])

        for m in range(max_allowed + 1):
            for e in used_edges:
                residual[e] -= m

            if all(residual[e] >= 0 for e in edge_ids):
                yield from backtrack(i + 1, current + [m])

            for e in used_edges:
                residual[e] += m

    yield from backtrack(0, [])

## 3) Rewrite `generate_multiplicity_collections`

### Old behavior
- It calls `generate_valid_multiplicities(...)` with scalar `N`.
- It may call `condition_fn(collection, values, N)` with a scalar assumption.

### New behavior
- Pass `edge_targets` dict into generator.
- Replace scalar condition check with `condition_fn_per_edge(...)`.
- Keep conjugacy logic unchanged.

In [ ]:
from collections import Counter

def generate_multiplicity_collections_v2(
    sequences,
    edge_ids,
    edge_targets,
    stop_early=False,
    max_multiplicity_path=None,
    edges=None,
    counterports=None,
    conjugacy_type=0,
    is_conjugate_fn=None,
):
    valid_collections = set()

    multiplicity_iter = generate_valid_multiplicities_v2(
        sequences=sequences,
        edge_targets=edge_targets,
        edge_ids=edge_ids,
        max_multiplicity_path=max_multiplicity_path,
    )

    valid_count = 0
    for multiplicities in multiplicity_iter:
        collection = Counter({
            seq: m for seq, m in zip(sequences, multiplicities) if m > 0
        })

        if not condition_fn_per_edge(collection, edge_ids, edge_targets):
            continue

        frozen = frozenset(collection.items())

        if conjugacy_type is not None and is_conjugate_fn is not None:
            if not is_conjugate_fn(frozen, edges, counterports, conjugacy_type):
                continue

        valid_collections.add(frozen)
        valid_count += 1

        if stop_early and valid_count >= stop_early:
            break

    return valid_collections

## 4) How `Topology.generate_solutions` Changes

The function flow remains the same. The only structural change is that you compute and pass `edge_targets` instead of scalar `N`.

1. Build candidate paths (`all_unique_sequences`, `is_valid_path`, loop pruning).
2. Normalize user input into `edge_targets` dict.
3. Generate multiplicity collections with the new per-edge solver.
4. Symmetry dedup with `find_unique_sets`.

This means your high-level behavior is preserved but generalized.

In [ ]:
# This is a drop-in style sketch for your Topology class.

def generate_solutions_v2(self, edge_targets_input, stop_early=False, default_edge_target=0):
    # 1) Generate candidate paths exactly as before
    all_seqs = all_unique_sequences(
        self.edges.keys(),
        min_length=self.power_set_min_length,
        max_length=self.power_set_max_length,
    )
    valid_paths = [s for s in all_seqs if self.is_valid_path(s)]
    valid_paths = self.prune_redundant_loops(valid_paths)

    edge_ids = list(self.edges.keys())

    # 2) Normalize heterogeneous targets
    edge_targets = normalize_edge_targets(
        edge_ids=edge_ids,
        edge_targets=edge_targets_input,
        default_edge_target=default_edge_target,
    )

    # 3) Multiplicity collections with per-edge constraints
    if self.conjugacy:
        valid_collections = generate_multiplicity_collections_v2(
            sequences=valid_paths,
            edge_ids=edge_ids,
            edge_targets=edge_targets,
            stop_early=stop_early,
            max_multiplicity_path=self.max_multiplicity,
            edges=self.edges,
            counterports=self.counterports,
            conjugacy_type=0,
            is_conjugate_fn=is_conjugate,
        )
    else:
        valid_collections = generate_multiplicity_collections_v2(
            sequences=valid_paths,
            edge_ids=edge_ids,
            edge_targets=edge_targets,
            stop_early=stop_early,
            max_multiplicity_path=self.max_multiplicity,
            edges=self.edges,
            counterports=self.counterports,
            conjugacy_type=None,
            is_conjugate_fn=None,
        )

    self.valid_collections = valid_collections
    self.solutions = self.find_unique_sets(valid_collections, self.symmetries, self.edges)
    return self.solutions

## 5) Minimal Sanity Example

Use this as a quick logic check before integrating into your full class:
- Suppose `edge_ids = [0,1,2,3]`.
- Targets: `{0:2, 1:3, 2:2, 3:3}`.
- Run `generate_valid_multiplicities_v2` on simple candidate paths and inspect whether each accepted multiplicity exactly satisfies edge totals.

In [ ]:
# Toy check (not tied to lattice file)
sequences = [
    (0, 1),
    (1, 2),
    (2, 3),
    (0, 3),
]
edge_ids = [0, 1, 2, 3]
targets = {0: 2, 1: 3, 2: 2, 3: 3}

all_mult = list(generate_valid_multiplicities_v2(
    sequences=sequences,
    edge_targets=targets,
    edge_ids=edge_ids,
    max_multiplicity_path=5,
))

print("Found", len(all_mult), "valid multiplicity vectors")
print("First few:", all_mult[:10])

## 6) Integration Notes (Important)

- Keep `self.N` only if other code still needs a scalar legacy mode.
- For heterogeneous mode, store `self.edge_targets` and derive strand-count logic from it.
- Anywhere you compare `count == N`, replace with `count == edge_targets[e]`.
- Anywhere you compute per-edge remaining capacity, always use the corresponding edge key `e`, never a hardcoded index.

If you want, the next step can be a second notebook section that maps each exact line in your current file to its replacement in a patch-style diff.